In [257]:
import pandas as pd
import numpy as np
import joblib

In [258]:
df = pd.read_csv('test_real_estate_samples(2).csv')

In [259]:
df.head()

,id,type,sub_type,start_date,end_date,listing_type,tom,building_age,total_floor_count,floor_no,room_count,size,address,furnished,heating_type,price,price_currency
0,49,Konut,Kooperatif,11/10/18,12/10/18,1,30,0,10,Müstakil,1+0,1.0,İzmir/Aliağa/Yenişakran,NaN,Fancoil,300000.0,TRY
1,506,Konut,Çiftlik Evi,10/25/18,2/23/19,1,121,2,2,NaN,2+2,250.0,İzmir/Foça/Yeniköy,NaN,Yok,1600000.0,TRY
2,2,Konut,Daire,2/13/19,NaN,1,14,0,20 ve üzeri,20 ve üzeri,1+0,43.0,İstanbul/Kartal/Kordonboyu,NaN,Fancoil,490000.0,TRY
3,528,Konut,Prefabrik Ev,11/16/18,1/21/19,1,66,NaN,1,NaN,3+1,122.0,Tekirdağ/Çerkezköy/Kızılpınar,NaN,Yok,103600.0,TRY
4,109,Konut,Kooperatif,11/10/18,1/9/19,1,60,0,10,Müstakil,1+0,1.0,İzmir/Aliağa/Yenişakran,NaN,Fancoil,300000.0,TRY


In [260]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,20.0,5257.250000,20427.688748,1.0,14.75,104.5,566.0,91731.0
listing_type,20.0,1.500000,0.512989,1.0,1.00,1.5,2.0,2.0
tom,20.0,55.200000,41.912942,0.0,30.00,38.0,92.5,152.0
size,17.0,165.588235,201.806857,1.0,85.00,125.0,160.0,905.0
furnished,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,20.0,380835.000000,640312.306223,750.0,3712.50,61300.0,492500.0,2450000.0


In [261]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Используем готовую модель из прошлой командной работы - Случайный лес

In [262]:
savedmodel = joblib.load('final_real_estate_model.pkl')
print("Модель загружена")

Модель загружена


In [263]:
print(savedmodel.keys())

dict_keys(['model', 'feature_names', 'scaler', 'continuous_cols'])


In [264]:
model = savedmodel['model']
expected_features = savedmodel['feature_names']
scaler = savedmodel['scaler']
continuous_cols = savedmodel['continuous_cols']

In [ ]:
X_test = df.drop(columns=['price'])
y_true = df['price'].astype(float)

KeyError: "['price'] not found in axis"

In [ ]:
X_test = pd.get_dummies(X_test, drop_first=True)

In [ ]:
X_test = X_test.reindex(columns=expected_features, fill_value=0)
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
if 'log1p' == 'log1p':
    y_pred = np.expm1(y_pred)
elif 'log1p' == 'log':
    y_pred = np.exp(y_pred)
else:
    y_pred = y_pred

print('После преобразования:', y_pred[:5].round(0))
print('Реальные цены:       ', y_true.values[:5])

После преобразования: [115272.  71177. 115640. 109608. 109608.]
Реальные цены:        [ 300000. 1600000.  490000.  103600.  300000.]


In [ ]:
mae_sk = mean_absolute_error(y_true, y_pred)
rmse_sk = np.sqrt(mean_squared_error(y_true, y_pred))
r2_sk = r2_score(y_true, y_pred)

In [ ]:
mae_man = np.mean(np.abs(y_true - y_pred))
rmse_man = np.sqrt(np.mean((y_true - y_pred)**2))
r2_man = 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))

In [ ]:
df['ФИО'] = "Ирхан Хабиров"
df['Группа'] = "23П-4"

In [ ]:
df['MAE'] = np.nan
df['RMSE'] = np.nan
df['R2'] = np.nan

In [ ]:
df.at[0, 'MAE'] = mae_sk
df.at[0, 'RMSE'] = rmse_sk
df.at[0, 'R2'] = r2_sk

In [ ]:
df['Predicted_Price'] = y_pred

In [ ]:
df.head()

,id,type,sub_type,start_date,end_date,listing_type,tom,building_age,total_floor_count,floor_no,...,furnished,heating_type,price,price_currency,ФИО,Группа,MAE,RMSE,R2,Predicted_Price
0,49,Konut,Kooperatif,11/10/18,12/10/18,1,30,0,10,Müstakil,...,NaN,Fancoil,300000.0,TRY,Ирхан Хабиров,23П-4,326398.860772,679072.087969,-0.183926,115271.651045
1,506,Konut,Çiftlik Evi,10/25/18,2/23/19,1,121,2,2,NaN,...,NaN,Yok,1600000.0,TRY,Ирхан Хабиров,23П-4,NaN,NaN,NaN,71176.837130
2,2,Konut,Daire,2/13/19,NaN,1,14,0,20 ve üzeri,20 ve üzeri,...,NaN,Fancoil,490000.0,TRY,Ирхан Хабиров,23П-4,NaN,NaN,NaN,115640.254971
3,528,Konut,Prefabrik Ev,11/16/18,1/21/19,1,66,NaN,1,NaN,...,NaN,Yok,103600.0,TRY,Ирхан Хабиров,23П-4,NaN,NaN,NaN,109608.100145
4,109,Konut,Kooperatif,11/10/18,1/9/19,1,60,0,10,Müstakil,...,NaN,Fancoil,300000.0,TRY,Ирхан Хабиров,23П-4,NaN,NaN,NaN,109608.100145


In [ ]:
print(f"MAE: {mae_sk:.2f} | RMSE: {rmse_sk:.2f} | R2: {r2_sk:.4f}")

MAE: 326398.86 | RMSE: 679072.09 | R2: -0.1839


По результатам могу сказать, что-то пошло не так. И модель не с могла показать высокую точность предсказаний. 

In [ ]:
df.to_csv('result_real_estate_samples.csv', index=False, encoding='utf-8-sig')
print("Файл сохранен.")

Файл сохранен.
